# Leveraging Pre-trained Models with Transfer Learning

## Part 3 of 3: Changing the last layer and tuning the whole model

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

torch.manual_seed(123)

- Loading a model from the PyTorch Hub: https://pytorch.org/docs/stable/hub.html

In [3]:
entrypoints = torch.hub.list('pytorch/vision:v0.13.0', force_reload=True)
for e in entrypoints:
    if "resnet" in e:
        print(e)

# %%
model = torch.hub.load('pytorch/vision:v0.13.0', 'resnet18', weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)  # all layers remain trainable
print(model)

Downloading: "https://github.com/pytorch/vision/zipball/v0.13.0" to /home/zeus/.cache/torch/hub/v0.13.0.zip
deeplabv3_resnet101
deeplabv3_resnet50
fcn_resnet101
fcn_resnet50
resnet101
resnet152
resnet18
resnet34
resnet50
wide_resnet101_2
wide_resnet50_2


Using cache found in /home/zeus/.cache/torch/hub/pytorch_vision_v0.13.0


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
# Alternatively:

#from torchvision.models import resnet18, ResNet18_Weights

#weights = ResNet18_Weights.IMAGENET1K_V1

## Custom Transform

- Also, we now have to keep in mind the preprocessing protocol that was used for pre-training the model:

In [4]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess_transform = weights.transforms()
print(preprocess_transform)

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [5]:
train_full   = torchvision.datasets.CIFAR10(root="data/", train=True,  download=True,  transform=preprocess_transform)
test_dataset = torchvision.datasets.CIFAR10(root="data/", train=False, download=True,  transform=preprocess_transform)

indices       = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(123))
train_dataset = Subset(train_full, indices[:45000])
val_dataset   = Subset(train_full, indices[45000:])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [6]:
def train_one_epoch(model, loader, optimizer, criterion, acc_metric, device):
    model.train()
    acc_metric.reset()
    total_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()


def evaluate(model, loader, criterion, acc_metric, device):
    model.eval()
    acc_metric.reset()
    total_loss = 0.0

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            acc_metric.update(logits, labels)

    return total_loss / len(loader.dataset), acc_metric.compute().item()

In [7]:
torch.manual_seed(123)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # lower lr for full fine-tuning
criterion = nn.CrossEntropyLoss()

train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

writer = SummaryWriter(log_dir="logs/resnet18-transfer-finetune")

dummy = torch.zeros(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy)


In [8]:
MAX_EPOCHS = 50

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, train_acc, device)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, val_acc, device)

    writer.add_scalars("Loss",     {"train": tr_loss, "val": vl_loss}, epoch)
    writer.add_scalars("Accuracy", {"train": tr_acc,  "val": vl_acc},  epoch)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

writer.close()

Epoch 01/50 | Train Loss: 0.3778  Acc: 0.8740 | Val Loss: 0.2059  Acc: 0.9308
Epoch 02/50 | Train Loss: 0.1174  Acc: 0.9625 | Val Loss: 0.2067  Acc: 0.9300
Epoch 03/50 | Train Loss: 0.0465  Acc: 0.9865 | Val Loss: 0.2195  Acc: 0.9316
Epoch 04/50 | Train Loss: 0.0397  Acc: 0.9880 | Val Loss: 0.2109  Acc: 0.9326
Epoch 05/50 | Train Loss: 0.0330  Acc: 0.9898 | Val Loss: 0.2150  Acc: 0.9362
Epoch 06/50 | Train Loss: 0.0302  Acc: 0.9907 | Val Loss: 0.1887  Acc: 0.9412
Epoch 07/50 | Train Loss: 0.0225  Acc: 0.9930 | Val Loss: 0.2227  Acc: 0.9326
Epoch 08/50 | Train Loss: 0.0269  Acc: 0.9910 | Val Loss: 0.2450  Acc: 0.9340
Epoch 09/50 | Train Loss: 0.0219  Acc: 0.9932 | Val Loss: 0.2331  Acc: 0.9344
Epoch 10/50 | Train Loss: 0.0230  Acc: 0.9928 | Val Loss: 0.1938  Acc: 0.9468
Epoch 11/50 | Train Loss: 0.0117  Acc: 0.9962 | Val Loss: 0.2376  Acc: 0.9374
Epoch 12/50 | Train Loss: 0.0206  Acc: 0.9928 | Val Loss: 0.2299  Acc: 0.9370
Epoch 13/50 | Train Loss: 0.0176  Acc: 0.9947 | Val Loss: 0.2417

In [ ]:
test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
te_loss, te_acc = evaluate(model, test_loader, criterion, test_acc, device)
print(f"\nTest Loss: {te_loss:.4f} | Test Accuracy: {te_acc:.4f}")